In [ ]:
%matplotlib widget
import io
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image

# FILE_PATH = "/home/metamobility3/Jinwoo/os_kinetics/0426_gain0p2_LG+SASD.npz"
FILE_PATH = "/home/metamobility3/Jinwoo/os_kinetics/0426_high_torque_SASD.npz"

In [16]:
data = np.load(FILE_PATH, allow_pickle=True)
print(f"Loaded : {os.path.basename(FILE_PATH)}")
print(f"Keys   : {list(data.keys())}")

time       = data["time"]
knee_angle = data["model_in_knee_angle_raw"]   # rad
knee_vel   = data["model_in_knee_vel_raw"]      # rad/s  (IMU-based)
model_out  = data["model_out_nmpkg"]            # Nm/kg
motor_torq = data["moment_raw"]                 # Nm

dt  = np.median(np.diff(time))
dur = time[-1] - time[0]
print(f"Duration: {dur:.1f} s  |  fs ≈ {1/dt:.1f} Hz  |  samples: {len(time)}")

Loaded : 0426_gain0p2_LG+SASD.npz
Keys   : ['time', 'knee_angle_r', 'knee_angle_l', 'knee_angle_r_u', 'knee_angle_l_u', 'knee_angle_r_u_gyr', 'knee_angle_l_u_gyr', 'gyro_thigh_r', 'gyro_shank_r', 'cmd_L', 'cmd_R', 'model_in_knee_angle_raw', 'model_in_knee_vel_raw', 'model_in_knee_angle_norm', 'model_in_knee_vel_norm', 'model_out_nmpkg', 'moment_raw', 'K_r', 'Soft_ctrl_r', 'K_l', 'Soft_ctrl_l', 'GPIO']
Duration: 170.3 s  |  fs ≈ 100.0 Hz  |  samples: 17035


In [17]:
PALETTE = {
    "angle" : "#2196F3",
    "vel"   : "#FF9800",
    "model" : "#9C27B0",
    "torque": "#F44336",
}

out = widgets.Output()

slider = widgets.FloatRangeSlider(
    value=[time[0], time[-1]],
    min=time[0], max=time[-1], step=0.1,
    description="Time (s):",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "initial"},
    continuous_update=False,
    readout_format=".1f",
)

def _draw(t0, t1):
    mask = (time >= t0) & (time <= t1)
    t_sel = time[mask]
    sigs = [
        (np.rad2deg(knee_angle[mask]), "Knee Angle (deg)",      PALETTE["angle"]),
        (np.rad2deg(knee_vel[mask]),   "Knee Ang. Vel. (deg/s)", PALETTE["vel"]),
        (model_out[mask],              "Model Output (Nm/kg)",   PALETTE["model"]),
        (motor_torq[mask],             "Motor Torque (Nm)",      PALETTE["torque"]),
    ]
    labels = [
        "Knee angle (model input)",
        "Knee ang. vel. — IMU",
        "Model output (Nm/kg)",
        "Actual motor torque (Nm)",
    ]
    fig, axs = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
    for i, (sig, ylabel, color) in enumerate(sigs):
        axs[i].plot(t_sel, sig, color=color, lw=1.5, label=labels[i])
        axs[i].set_ylabel(ylabel, fontsize=13)
        axs[i].spines[["top", "right"]].set_visible(False)
        axs[i].tick_params(labelsize=11)
        axs[i].legend(fontsize=11, loc="upper right")
        if i > 0:
            axs[i].axhline(0, color="gray", lw=0.8, ls="--")
    axs[-1].set_xlabel("Time (s)", fontsize=13)
    fig.suptitle(os.path.basename(FILE_PATH), fontsize=13, y=1.01)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    out.clear_output(wait=True)
    with out:
        display(Image(data=buf.read()))

def _on_slider(change):
    _draw(*slider.value)

slider.observe(_on_slider, names="value")
display(slider, out)
_draw(time[0], time[-1])

FloatRangeSlider(value=(0.01019520400041074, 170.3500745890001), continuous_update=False, description='Time (s…

Output()

In [18]:
# ── summary statistics for the current slider window ─────────────────────────
t0_s, t1_s = slider.value
mask_s = (time >= t0_s) & (time <= t1_s)

signals = {
    "Knee angle (deg)"       : np.rad2deg(knee_angle[mask_s]),
    "Knee ang. vel. (deg/s)" : np.rad2deg(knee_vel[mask_s]),
    "Model output (Nm/kg)"   : model_out[mask_s],
    "Motor torque (Nm)"      : motor_torq[mask_s],
}

print(f"{'Signal':<28} {'min':>8} {'max':>8} {'mean':>8} {'std':>8}")
print("-" * 62)
for name, sig in signals.items():
    print(f"{name:<28} {sig.min():8.3f} {sig.max():8.3f} {sig.mean():8.3f} {sig.std():8.3f}")

Signal                            min      max     mean      std
--------------------------------------------------------------
Knee angle (deg)              -94.100    2.400  -27.895   23.376
Knee ang. vel. (deg/s)       -561.524  480.335   -0.223  184.598
Model output (Nm/kg)           -0.824    0.215   -0.003    0.129
Motor torque (Nm)             -72.471   18.902   -0.227   11.339
